In [13]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import io

## RocksDB

In [23]:
# Throughput

n_rand_read = """
cores, mode, slowdown
1, kernel_rr, 1.09
2, kernel_rr, 1.26
4, kernel_rr, 1.37
8, kernel_rr, 1.78
16, kernel_rr, 1.90
32, kernel_rr, 2.29
1, whole_system_rr, 1.15
2, whole_system_rr, 3.14
4, whole_system_rr, 5.25
8, whole_system_rr, 8.97
16, whole_system_rr, 27.69
32, whole_system_rr, 33.11"""

n_seq_read = """
cores, mode, slowdown,
1, kernel_rr, 1.34
2, kernel_rr, 1.67
4, kernel_rr, 2.03
8, kernel_rr, 2.79
16, kernel_rr, 4.11
32, kernel_rr, 6.90
1, whole_system_rr, 1.27
2, whole_system_rr, 4.93
4, whole_system_rr, 11.76
8, whole_system_rr, 29.94
16, whole_system_rr, 97.07
32, whole_system_rr, 759.08"""

n_rand_read_one_write = """
cores, mode, slowdown
1, kernel_rr, 1.10
2, kernel_rr, 1.01
4, kernel_rr, 1.06
8, kernel_rr, 1.52
16, kernel_rr, 3.19
32, kernel_rr, 6.91
1, whole_system_rr, 1.08
2, whole_system_rr, 2.96
4, whole_system_rr, 5.88
8, whole_system_rr, 12.29
16, whole_system_rr, 24.92
32, whole_system_rr, 53.66"""

n_rand_read_one_scan = """
cores, mode, slowdown
1, kernel_rr, 1.12
2, kernel_rr, 1.07
4, kernel_rr, 1.38
8, kernel_rr, 1.80
16, kernel_rr, 1.90
32, kernel_rr, 2.25
1, whole_system_rr, 1.11
2, whole_system_rr, 2.71
4, whole_system_rr, 5.08
8, whole_system_rr, 9.49
16, whole_system_rr, 28.11
32, whole_system_rr, 34.70"""

n_rand_seeker = """
cores, mode, slowdown
1, kernel_rr, 1.08
2, kernel_rr, 1.18
4, kernel_rr, 1.35
8, kernel_rr, 2.03
16, kernel_rr, 2.42
32, kernel_rr, 2.77
1, whole_system_rr, 1.12
2, whole_system_rr, 2.92
4, whole_system_rr, 5.58
8, whole_system_rr, 10.24
16, whole_system_rr, 29.70
32, whole_system_rr, 48.16"""

## Kernel Build

In [7]:
my_results = pd.read_csv("kernel_build/kernel_build-kernel_build-time.csv")
my_results.drop("trial", inplace=True, axis = 1)
my_results

,cores,mode,value
0,1,baseline,1616.789
1,2,baseline,845.913
2,4,baseline,446.491
3,8,baseline,248.667
4,16,baseline,150.382
5,32,baseline,98.678
6,1,kernel_rr,1875.859
7,2,kernel_rr,1041.084
8,4,kernel_rr,573.586
9,8,kernel_rr,392.735


In [10]:
baseline = my_results[my_results['mode'] == 'baseline'].set_index('cores')['value']

cores
1     1616.789
2      845.913
4      446.491
8      248.667
16     150.382
32      98.678
Name: value, dtype: float64

In [12]:
results = []
for mode in ['kernel_rr', 'whole_system_rr']:
    mode_data = my_results[my_results['mode'] == mode].copy()
    mode_data['slowdown'] = mode_data.apply(
        lambda row: row['value'] / baseline[row['cores']], axis=1
    )
    results.append(mode_data[['cores', 'mode', 'value', 'slowdown']])

slowdown_df = pd.concat(results, ignore_index=True)
slowdown_df

,cores,mode,value,slowdown
0,1,kernel_rr,1875.859,1.160237
1,2,kernel_rr,1041.084,1.230722
2,4,kernel_rr,573.586,1.284653
3,8,kernel_rr,392.735,1.579361
4,16,kernel_rr,552.983,3.677189
5,32,kernel_rr,915.837,9.281066
6,1,whole_system_rr,1807.693,1.118076
7,2,whole_system_rr,2771.263,3.276061
8,4,whole_system_rr,2790.494,6.249833
9,8,whole_system_rr,2960.284,11.904611


In [16]:
paper_results = """
cores, mode, slowdown
1, kernel_rr, 1.15
2, kernel_rr, 1.22
4, kernel_rr, 1.26
8, kernel_rr, 1.56
16, kernel_rr, 3.56
32, kernel_rr, 8.68
1, whole_system_rr, 1.11
2, whole_system_rr, 3.26
4, whole_system_rr, 6.27
8, whole_system_rr, 11.47
16, whole_system_rr, 20.62
32, whole_system_rr, 37.20
"""

paper_df = pd.read_csv(io.StringIO(paper_results.strip()), skipinitialspace=True)
paper_df

,cores,mode,slowdown
0,1,kernel_rr,1.15
1,2,kernel_rr,1.22
2,4,kernel_rr,1.26
3,8,kernel_rr,1.56
4,16,kernel_rr,3.56
5,32,kernel_rr,8.68
6,1,whole_system_rr,1.11
7,2,whole_system_rr,3.26
8,4,whole_system_rr,6.27
9,8,whole_system_rr,11.47


In [19]:
percent_change = slowdown_df.copy()
percent_change["paper_slowdown"] = paper_df["slowdown"]
percent_change["percent_change"] = ((percent_change["slowdown"].round(2)
                                         - percent_change["paper_slowdown"])
                                        / percent_change["paper_slowdown"])
percent_change

,cores,mode,value,slowdown,paper_slowdown,percent_change
0,1,kernel_rr,1875.859,1.160237,1.15,0.008696
1,2,kernel_rr,1041.084,1.230722,1.22,0.008197
2,4,kernel_rr,573.586,1.284653,1.26,0.015873
3,8,kernel_rr,392.735,1.579361,1.56,0.012821
4,16,kernel_rr,552.983,3.677189,3.56,0.033708
5,32,kernel_rr,915.837,9.281066,8.68,0.069124
6,1,whole_system_rr,1807.693,1.118076,1.11,0.009009
7,2,whole_system_rr,2771.263,3.276061,3.26,0.006135
8,4,whole_system_rr,2790.494,6.249833,6.27,-0.003190
9,8,whole_system_rr,2960.284,11.904611,11.47,0.037489


## DPDK

### Redis

### Nginx